# ☕ BunaDoctor — Dataset Verification

### Count every image **before** any preprocessing

**Project** · Design and Development of a Coffee Leaf Disease Diagnosis System Using Deep Learning
**Team** · Addis Ababa University, Department of Computer Science · FYP I, 2026/27
**Owner** · Hayredin Mohammed (UGR/2361/16) · Week 3
**Companion report** · `docs/12-DATASET-VERIFICATION.md`

---

This notebook answers one question, and answers it with files rather than with citations:

> **How many coffee-leaf images do we actually have, per class, before we resize, deduplicate, augment or split anything?**

It runs on **Google Colab** and on **Kaggle** with no edits. It detects where it is, finds the
datasets that are already mounted, downloads the ones that can be downloaded without a login,
unpacks them, counts every image file, compares the count with the published claim, and writes a
signed-off table plus a dashboard.

| What it produces | File |
|---|---|
| Machine-readable counts | `out/dataset_counts.json` |
| One row per dataset | `out/dataset_counts.csv` |
| One row per dataset × class | `out/dataset_class_counts.csv` |
| Paste-ready markdown | `out/dataset_counts.md` |
| Dashboard | `out/dashboard.png` |
| Claimed vs observed | `out/claimed_vs_observed.png` |
| Everything zipped | `out/buna-verification.zip` |

> **A count is only a count if it is reproducible.** Every number below is a `len()` over a real
> directory. If a file is not on disk, the notebook says **NOT PRESENT** — it never fills the gap
> with a number from a paper.

## How to run this

### The 30-second version — no download, no account

1. Open the notebook.
2. **Runtime → Run all** (Colab) or **Run All** (Kaggle).
3. Leave `SETTINGS["download"] = False`.

You will get the **self-test** (proves the counter is correct), the **probe** (proves every record
still exists), and a dashboard where every dataset says **NOT PRESENT**. That is the expected
result on a fresh machine, and it takes about 30 seconds.

### Colab — fetching the data

1. **Runtime → Change runtime type → CPU.** No GPU is needed to count files.
2. Open cell **3 · SETTINGS** and set `"download": True`.
3. Kaggle datasets need a token: on kaggle.com go to **Account → Create New API Token**, then
   upload the downloaded `kaggle.json` when the notebook asks, or paste it into
   `SETTINGS["kaggle_json"]`.
4. **Runtime → Run all.**

### Kaggle — fetching the data

1. Create a notebook and choose **Add Input → Datasets**.
2. Add the datasets you want by slug (the table is in `notebooks/README.md`). They appear under
   `/kaggle/input/` and the notebook finds them automatically — **no token needed**.
3. **Run all.**

### Datasets that need a browser

**Mendeley Data does not allow anonymous downloads.** The public API lists every file with its
size and SHA-256, but the bytes require a logged-in session. For those records the notebook prints
the exact page, the file names and the sizes, and waits for you to drop the archive into
`raw/<key>/`. Everything else is automatic.

| Record | Why it needs a browser |
|---|---|
| JMuBEN, JMuBEN2, BRACOL, Saposoa, Silva, RoCoLe, Uganda | Mendeley login |
| CoffeeLeaf-CO v2 | **none** — direct Zenodo link, downloaded automatically |
| Ethiopian, Uganda-augmented, JMuBEN mirror, RoCoLe mirror, Silva mirror | **none** — Kaggle CLI |

> **Rule:** a Kaggle *mirror* is for convenience only. The count is reported against the original
> record, and the licence is always the publisher's, never the mirror's.

---
## 1 · Bootstrap

Detect the environment, install the three or four extras, and create the workspace. Nothing here
touches the network except `pip`, and every failure is non-fatal.

In [ ]:
import os, sys, io, json, re, csv, time, math, shutil, hashlib, platform
import subprocess, importlib, importlib.util, importlib.metadata
import tempfile, glob, random, zipfile, tarfile
from pathlib import Path
from collections import Counter, defaultdict

def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "--disable-pip-version-check", *pkgs],
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)

for _mod, _pkg in [("PIL", "pillow"), ("pandas", "pandas"),
                   ("requests", "requests"), ("matplotlib", "matplotlib")]:
    if importlib.util.find_spec(_mod) is None:
        _pip(_pkg)

IS_COLAB  = "google.colab" in sys.modules or os.path.isdir("/content")
IS_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or os.path.isdir("/kaggle/input")
ENV       = "Google Colab" if IS_COLAB else ("Kaggle" if IS_KAGGLE else "Local")

BASE = (Path("/content/buna-verify") if IS_COLAB
        else Path("/kaggle/working/buna-verify") if IS_KAGGLE
        else Path.cwd() / "buna-verify")
RAW       = BASE / "raw"
EXTRACTED = BASE / "extracted"
OUT       = BASE / "out"
for _d in (RAW, EXTRACTED, OUT):
    _d.mkdir(parents=True, exist_ok=True)

if IS_COLAB:   # unrar is only needed for the Saposoa archive
    subprocess.run("apt-get -qq install -y unrar-free >/dev/null 2>&1 || "
                   "apt-get -qq install -y unrar >/dev/null 2>&1 || true",
                   shell=True, check=False)

print("""
+----------------------------------------------------------------------+
|  BunaDoctor . dataset verification                                   |
|  count every image BEFORE any preprocessing                          |
+----------------------------------------------------------------------+""")
print(f"  environment : {ENV}")
print(f"  python      : {platform.python_version()}")
print(f"  workspace   : {BASE}")
print(f"  raw         : {RAW}")
print(f"  out         : {OUT}")

---
## 2 · SETTINGS — the only cell you edit

Everything else is automatic. Change one value at a time and re-run from here.

In [ ]:
SETTINGS = {
    # ── what to fetch ──────────────────────────────────────────────────────
    "download": False,        # True = fetch everything that can be fetched without a login
    "datasets": "all",        # "all", or a list: ["ethiopian", "uganda_aug", "coffeeleaf_co"]

    # ── optional, slower work ──────────────────────────────────────────────
    "run_phash": False,       # perceptual-hash duplicate estimate (minutes, not seconds)
    "phash_datasets": ["uganda", "rocole", "ethiopian"],
    "dimension_sample": 300,  # how many images per set to open for width x height

    # ── credentials (Colab only, and only for Kaggle downloads) ────────────
    "kaggle_json": "",        # paste {"username": "...", "key": "..."} here if you prefer
    "interactive_upload": False,  # Colab: offer a file picker for a missing archive
}

print(f"  download        : {SETTINGS['download']}")
print(f"  datasets        : {SETTINGS['datasets']}")
print(f"  perceptual hash : {SETTINGS['run_phash']}")
print("  (the active set is computed in the next cell, once the registry exists)")

---
## 3 · The registry

One row per **record**, not per paper. A record is identified by its DOI or its Zenodo ID, because
that is the thing that can be re-fetched and re-counted.

`claim_total` is what the publisher says. `expected` is the per-class split **where the publisher
publishes one**; where it does not, the value is `None` and the notebook reports the total only.
`tier` is the evidence level of the count we can defend today:

| Tier | Meaning |
|---|---|
| **T1** | verified against **two independent** releases |
| **T2** | verified from **one** publisher file manifest |
| **T3** | published figure, archive **not opened** |
| **T4** | claimed only |

In [ ]:
CLASS_ORDER = ["healthy", "rust", "cercospora", "miner", "phoma"]

# The licence of the *record*, not of the mirror. A mirror relabels; the DOI does not.
LICENCES = {
    "jmuben": "CC BY 4.0", "jmuben2": "CC BY 4.0",
    "jmuben_kaggle": "Unknown (mirror)",
    "bracol": "CC BY 4.0",
    "ethiopian": "CC0 1.0",
    "uganda_aug": "CC BY-NC-SA 4.0",
    "uganda": "CC BY 4.0",
    "rocole": "CC BY 4.0",
    "silva": "CC BY 4.0",
    "saposoa": "CC BY 4.0",
    "coffeeleaf_co": "CC BY 4.0",
}

DATASETS = [
    # ── the training corpus ───────────────────────────────────────────────
    dict(key="jmuben", label="JMuBEN", role="training", tier="T2",
         record="Mendeley t2r6rszp5c v1", doi="10.17632/t2r6rszp5c.1",
         page="https://data.mendeley.com/datasets/t2r6rszp5c/1",
         probe="https://data.mendeley.com/public-api/datasets/t2r6rszp5c",
         country="Kenya", species="Arabica",
         claim_total=22591, claim_note="Rust + Cercospora + Phoma",
         expected={"cercospora": 7681, "rust": 8336, "phoma": 6571},
         expected_note="per-class split of the combined 58,549; the three-class sum is 22,588",
         mendeley_files=["Cerscospora-20210326T085017Z-001.zip (252.2 MB)",
                         "Leaf rust-20210326T083416Z-001.zip (69.5 MB)",
                         "Phoma-20210326T082051Z-001.zip (227.5 MB)"]),

    dict(key="jmuben2", label="JMuBEN2", role="training", tier="T2",
         record="Mendeley tgv3zb82nd v1", doi="10.17632/tgv3zb82nd.1",
         page="https://data.mendeley.com/datasets/tgv3zb82nd/1",
         probe="https://data.mendeley.com/public-api/datasets/tgv3zb82nd",
         country="Kenya", species="Arabica",
         claim_total=35964, claim_note="Healthy + Leaf Miner",
         expected={"healthy": 18983, "miner": 16978},
         expected_note="per-class split of the combined 58,549; the two-class sum is 35,961",
         mendeley_files=["Healthy-20210326T083815Z-001.zip (566.9 MB)",
                         "Miner-20210326T082341Z-001.zip (724.8 MB)"]),

    dict(key="jmuben_kaggle", label="JMuBEN (Kaggle mirror)", role="mirror", tier="T1",
         record="Kaggle noamaanabdulazeem/jmuben-coffee-dataset", doi="",
         page="https://www.kaggle.com/datasets/noamaanabdulazeem/jmuben-coffee-dataset",
         probe="https://www.kaggle.com/api/v1/datasets/view/noamaanabdulazeem/jmuben-coffee-dataset",
         country="Kenya", species="Arabica",
         claim_total=58555, claim_note="the paper's combined JMuBEN + JMuBEN2 figure",
         expected={"cercospora": 7681, "healthy": 18983, "rust": 8336, "miner": 16978, "phoma": 6571},
         expected_note="two independent mirrors agree on 58,549; the paper says 58,555",
         kaggle="noamaanabdulazeem/jmuben-coffee-dataset",
         kaggle_note="contains both JMuBEN and JMuBEN2; do not add to the Mendeley rows"),

    dict(key="bracol", label="BRACOL", role="training", tier="T3",
         record="Mendeley yy2k5y8mxg v1", doi="10.17632/yy2k5y8mxg.1",
         page="https://data.mendeley.com/datasets/yy2k5y8mxg/1",
         probe="https://data.mendeley.com/public-api/datasets/yy2k5y8mxg",
         country="Brazil", species="Arabica",
         claim_total=3894, claim_note="1,747 whole leaves + 2,147 symptom crops",
         expected={},
         expected_note="labels live in the annotation files, not in folder names",
         mendeley_files=["BRACOL_coffee_leaf_ images_datasets.zip (164.5 MB)"]),

    # ── external evaluation sets ──────────────────────────────────────────
    dict(key="ethiopian", label="Ethiopian (Kaggle)", role="external", tier="T2",
         record="Kaggle biniyamyoseph/ethiopian-coffee-leaf-disease", doi="",
         page="https://www.kaggle.com/datasets/biniyamyoseph/ethiopian-coffee-leaf-disease",
         probe="https://www.kaggle.com/api/v1/datasets/view/biniyamyoseph/ethiopian-coffee-leaf-disease",
         country="Ethiopia", species="Arabica",
         claim_total=12000, claim_note="train aug/ 2,700 x 4 + test/ 300 x 4",
         expected={"cercospora": 3000, "healthy": 3000, "rust": 3000, "phoma": 3000},
         expected_note="3,000 per class; 2,700 augmented train + 300 clean test",
         kaggle="biniyamyoseph/ethiopian-coffee-leaf-disease"),

    dict(key="uganda_aug", label="Uganda augmented (Kaggle)", role="external", tier="T2",
         record="Kaggle chelangatspecioza/coffeedataset34", doi="",
         page="https://www.kaggle.com/datasets/chelangatspecioza/coffeedataset34",
         probe="https://www.kaggle.com/api/v1/datasets/view/chelangatspecioza/coffeedataset34",
         country="Uganda", species="Arabica",
         claim_total=26315, claim_note="the uploader's own augmented release",
         expected={"healthy": 9045, "rust": 8330, "phoma": 8960},
         expected_note="counted from the Kaggle manifest; claim is 26,315",
         kaggle="chelangatspecioza/coffeedataset34"),

    dict(key="uganda", label="Uganda (Mendeley)", role="external", tier="T2",
         record="Mendeley k36wnd6knb v1", doi="10.17632/k36wnd6knb.1",
         page="https://data.mendeley.com/datasets/k36wnd6knb/1",
         probe="https://data.mendeley.com/public-api/datasets/k36wnd6knb",
         country="Uganda", species="Arabica",
         claim_total=3312, claim_note="the description's figure",
         expected={"healthy": 1179, "rust": 1033, "phoma": 1110},
         expected_note="class is encoded in the filename prefix: 1_ / 1200_ / 2300_",
         filename_rules=[(r"^1200_", "rust"), (r"^2300_", "phoma"), (r"^1_", "healthy")],
         mendeley_files=["3,322 individual JPG files (25.4 MB total)"]),

    dict(key="rocole", label="RoCoLe", role="external", tier="T1",
         record="Mendeley c5yvn32dzg v2", doi="10.17632/c5yvn32dzg.2",
         page="https://data.mendeley.com/datasets/c5yvn32dzg/2",
         probe="https://data.mendeley.com/public-api/datasets/c5yvn32dzg",
         country="Ecuador", species="Robusta",
         claim_total=1560, claim_note="390 plants x 4 views",
         expected={"rust": 779, "healthy": 780},
         expected_note="E = enferma (rust), H = sana (healthy); one stray C11P11HE.jpg",
         filename_rules=[(r"E\d*\.jpe?g$", "rust"), (r"H\d*\.jpe?g$", "healthy")],
         group_regex=r"C(\d+)P(\d+)", group_expected=390,
         kaggle="anitagarg123/coffee-leaves",
         kaggle_note="2.27 GB mirror that includes RoCoLE-csv.csv; cite the Mendeley DOI",
         mendeley_files=["1,560 JPG + annotations", "RoCoLe-voc.tar.gz (697.6 MB)"]),

    dict(key="silva", label="Silva 2020", role="external", tier="T2",
         record="Mendeley vfxf4trtcg v5", doi="10.17632/vfxf4trtcg.5",
         page="https://data.mendeley.com/datasets/vfxf4trtcg/5",
         probe="https://data.mendeley.com/public-api/datasets/vfxf4trtcg",
         country="Brazil", species="Arabica",
         claim_total=542, claim_note="285 rust + 257 miner",
         expected={"rust": 285, "miner": 257},
         expected_note="confirmed by the same author's Kaggle upload, so T2 not T1",
         kaggle="alvarole/coffee-leaves-disease",
         kaggle_note="542 JPG + 542 XML, uploaded by the same author",
         mendeley_files=["miner_img_xml.zip (667.6 MB)", "rust_xml_image.zip (828.3 MB)"]),

    dict(key="saposoa", label="Saposoa, Peru", role="external", tier="T3",
         record="Mendeley mfpxg4y65r v2", doi="10.17632/mfpxg4y65r.2",
         page="https://data.mendeley.com/datasets/mfpxg4y65r/2",
         probe="https://data.mendeley.com/public-api/datasets/mfpxg4y65r",
         country="Peru", species="Arabica",
         claim_total=1500, claim_note="rust + healthy (+ Mycena)",
         expected={"rust": None, "healthy": None},
         expected_note="no per-class figure published",
         mendeley_files=["Coffe Leaf.rar (3,448.4 MB)"]),

    dict(key="coffeeleaf_co", label="CoffeeLeaf-CO v2", role="external", tier="T3",
         record="Zenodo 22756170", doi="10.5281/zenodo.22756170",
         page="https://zenodo.org/records/22756170",
         probe="https://zenodo.org/api/records/22756170",
         country="Colombia", species="Arabica",
         claim_total=2315, claim_note="rust + miner (+ weevil)",
         expected={"rust": None, "miner": None},
         expected_note="detection annotations; no per-class figure published",
         http=["https://zenodo.org/records/22756170/files/CoffeeLeaf-CO-v2.zip?download=1"]),
]

print(f"  registry: {len(DATASETS)} records")
for m in DATASETS:
    m["licence"] = LICENCES.get(m["key"], "see record")
    star = {"training": "*", "external": "+", "mirror": "="}[m["role"]]
    flag = "  <-- NON-COMMERCIAL" if "NC" in m["licence"].replace(" ", "-").upper() else ""
    print(f"   {star} {m['key']:<15} {m['tier']:<3} {m['claim_total']:>7,}  "
          f"{m['label']:<26} {m['licence']}{flag}")
print("   * training   + external   = mirror (never added to a total)")

_nc = [m["label"] for m in DATASETS if "NC" in m["licence"].replace(" ", "-").upper()]
if _nc:
    print(f"\n  LICENCE FLAG: {', '.join(_nc)} is non-commercial. The Mendeley original")
    print("  (CC BY 4.0) is the one to train on if BunaDoctor is to be a company.")

ACTIVE = [m for m in DATASETS if SETTINGS["datasets"] == "all" or m["key"] in SETTINGS["datasets"]]
print(f"\n  active: {len(ACTIVE)} of {len(DATASETS)}")

---
## 4 · Helpers

`canon_label` maps the many spellings in the wild — `Cerscospora`, `Health leaves`, `Leaf rust`,
`miner_img_xml` — onto the five canonical classes. `probe` is a cheap existence check. `sha256` is
the one line a reviewer can re-run.

In [ ]:
from PIL import Image

ALIASES = {
    "healthy": "healthy", "health": "healthy", "health leaves": "healthy",
    "health_leaf": "healthy", "sana": "healthy", "h": "healthy",
    "rust": "rust", "leaf rust": "rust", "leaf_rust": "rust", "coffee leaf rust": "rust",
    "clr": "rust", "roya": "rust", "enferma": "rust", "e": "rust",
    "cercospora": "cercospora", "cerscospora": "cercospora", "cercospora leaf spot": "cercospora",
    "brown eye spot": "cercospora", "brown_eye_spot": "cercospora", "coffee_browneye": "cercospora",
    "miner": "miner", "leaf miner": "miner", "leaf_miner": "miner", "minador": "miner",
    "phoma": "phoma", "phoma leaf spot": "phoma",
    "red spider mite": "red_spider_mite", "mite": "red_spider_mite",
    "mycena": "mycena", "weevil": "weevil",
}

def canon_label(text):
    """Map a folder or file name onto a canonical class, or None."""
    t = re.sub(r"[_\-]+", " ", str(text)).strip().lower()
    t = re.sub(r"\s+", " ", t)
    if t in ALIASES:
        return ALIASES[t]
    # substring fallback: "miner_img_xml", "Leaf rust 2021", "Cerscospora-..."
    for needle, cls in (("cercospora", "cercospora"), ("cerscospora", "cercospora"),
                        ("browneye", "cercospora"), ("brown eye", "cercospora"),
                        ("miner", "miner"), ("phoma", "phoma"),
                        ("healthy", "healthy"), ("health", "healthy"),
                        ("rust", "rust")):
        if needle in t:
            return cls
    return None

SPLIT_ALIASES = {
    "train": "train", "training": "train", "train aug": "train", "trainaug": "train",
    "aug": "train", "augmented": "train",
    "test": "test", "testing": "test", "holdout": "test",
    "val": "val", "valid": "val", "validation": "val", "dev": "val",
}

IMAGE_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp", ".gif"}

def human(n):
    return f"{n:,}"

def probe(url, timeout=25):
    """Cheap existence check. A failed probe is a finding, not an error."""
    t0 = time.time()
    try:
        import requests
        r = requests.get(url, timeout=timeout, headers={"accept": "application/json"})
        return {"url": url, "status": r.status_code, "bytes": len(r.content),
                "ms": int((time.time() - t0) * 1000), "ok": r.status_code < 400}
    except Exception as e:
        return {"url": url, "status": None, "bytes": 0,
                "ms": int((time.time() - t0) * 1000), "ok": False,
                "error": str(e)[:160]}

def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

def text_table(rows, headers):
    widths = [len(h) for h in headers]
    for row in rows:
        for i, cell in enumerate(row):
            widths[i] = max(widths[i], len(str(cell)))
    line = "  ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
    out = [line, "  ".join("-" * w for w in widths)]
    for row in rows:
        out.append("  ".join(str(c).ljust(widths[i]) for i, c in enumerate(row)))
    return "\n".join(out)

print("  helpers ready")

---
## 5 · `count_tree` — the actual deliverable

Walks a directory and counts **files as they are**. No resize, no dedup, no cleaning, no split.

The class of an image is decided in this order:

1. an explicit **filename rule** for the dataset (`^1200_` → rust for Uganda; `E1.jpg` → rust for RoCoLe),
2. the **nearest ancestor folder** that names a class (`.../Cerscospora/4 (1).jpg`),
3. a class word inside the **filename** itself,
4. otherwise `unlabelled` — and `unlabelled` is reported, never hidden.

The split is read from any ancestor folder called `train`, `test`, `val` … and `train aug` is
normalised to `train`. A group regex, where one exists, counts **plants** rather than images — that
is the number the leakage-safe split needs.

In [ ]:
def count_tree(root, meta=None, sample_dims=300):
    """Count image files under `root`. Returns a plain dict, JSON-serialisable."""
    root = Path(root)
    meta = meta or {}
    rules = meta.get("filename_rules") or []
    group_re = meta.get("group_regex")

    by_class   = Counter()
    by_split   = Counter()          # canonical: train / test / val / unspecified
    by_split_class = defaultdict(Counter)
    by_ext     = Counter()
    dims       = []
    groups     = set()
    files = total_bytes = unreadable = 0
    top_dirs = Counter()

    for p in root.rglob("*"):
        if not p.is_file() or p.suffix.lower() not in IMAGE_EXT:
            continue
        files += 1
        try:
            total_bytes += p.stat().st_size
        except OSError:
            pass

        rel = p.relative_to(root)
        parts = [x for x in rel.parts[:-1]]
        if parts:
            top_dirs[parts[0]] += 1

        # 1. explicit filename rule
        cls = None
        for pattern, value in rules:
            if re.search(pattern, p.name, re.I):
                cls = value
                break
        # 2. nearest class folder
        if cls is None:
            for part in reversed(parts):
                c = canon_label(part)
                if c:
                    cls = c
                    break
        # 3. class word in the filename
        if cls is None:
            cls = canon_label(p.stem)
        if cls is None:
            cls = "unlabelled"

        # split
        split = "unspecified"
        for part in parts:
            key = re.sub(r"[_\-]+", " ", part).strip().lower()
            key = re.sub(r"\s+", " ", key)
            if key in SPLIT_ALIASES:
                split = SPLIT_ALIASES[key]
                break

        by_class[cls] += 1
        by_split[split] += 1
        by_split_class[split][cls] += 1
        by_ext[p.suffix.lower()] += 1

        if group_re:
            m = re.search(group_re, p.stem)
            if m:
                groups.add("P".join(m.groups()))

        if len(dims) < sample_dims:
            try:
                with Image.open(p) as im:
                    dims.append(im.size)
            except Exception:
                unreadable += 1

    return {
        "root": str(root),
        "images": files,
        "bytes": total_bytes,
        "by_class": dict(sorted(by_class.items(), key=lambda kv: -kv[1])),
        "by_split": dict(by_split),
        "by_split_class": {k: dict(v) for k, v in by_split_class.items()},
        "by_ext": dict(by_ext),
        "dimensions": dims,
        "unreadable": unreadable,
        "groups": len(groups) if group_re else None,
        "top_dirs": dict(top_dirs.most_common(12)),
    }

print("  count_tree defined")

---
## 6 · Self-test — prove the counter before trusting it

A counter that has never been checked is a guess. This builds a synthetic tree with a known answer,
counts it, asserts every number, and deletes it. It runs in under a second and it runs **every
time**, so the first thing you see is `SELF-TEST PASS`.

In [ ]:
def _self_test():
    tmp = Path(tempfile.mkdtemp(prefix="buna-selftest-"))
    try:
        spec = {
            "train": {"Healthy": 3, "Leaf rust": 2, "Cerscospora": 1},
            "test":  {"Healthy": 1, "Leaf miner": 2},
        }
        for split, classes in spec.items():
            for cls, n in classes.items():
                d = tmp / split / cls
                d.mkdir(parents=True, exist_ok=True)
                for i in range(n):
                    Image.new("RGB", (24, 16), (i * 20 % 255, 90, 140)).save(d / f"{i}.jpg")

        got = count_tree(tmp, meta={})
        assert got["images"] == 9, got["images"]
        assert got["by_class"]["healthy"] == 4, got["by_class"]
        assert got["by_class"]["rust"] == 2, got["by_class"]
        assert got["by_class"]["cercospora"] == 1, got["by_class"]
        assert got["by_class"]["miner"] == 2, got["by_class"]
        assert got["by_split"]["train"] == 6, got["by_split"]
        assert got["by_split"]["test"] == 3, got["by_split"]
        assert got["dimensions"][0] == (24, 16), got["dimensions"][:1]

        # filename-rule path (the Uganda / RoCoLe shape: flat files, label in the name)
        flat = tmp / "flat"
        flat.mkdir()
        for name in ["1_0.jpg", "1_1.jpg", "1200_7.jpg", "2300_3.jpg"]:
            Image.new("RGB", (8, 8)).save(flat / name)
        got2 = count_tree(flat, meta={"filename_rules": [(r"^1200_", "rust"),
                                                          (r"^2300_", "phoma"),
                                                          (r"^1_", "healthy")]})
        assert got2["by_class"] == {"healthy": 2, "rust": 1, "phoma": 1}, got2["by_class"]

        return True, "9 synthetic images, 2 layouts, every count matched"
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

_ok, _msg = _self_test()
print(("  SELF-TEST PASS  " if _ok else "  SELF-TEST FAIL  ") + _msg)
assert _ok

---
## 7 · Probe — never cite a record you have not checked today

One cheap request per record. It proves the DOI, the Zenodo ID and the Kaggle slug still resolve on
the day the report is written. A probe that fails is recorded as a finding and the notebook
continues; nothing here is allowed to stop the run.

In [ ]:
probes = {}
print(f"{'record':<34} {'status':>7}  {'ms':>6}  result")
print("-" * 78)
for meta in DATASETS:
    url = meta.get("probe")
    if not url:
        probes[meta["key"]] = {"ok": None, "status": None, "note": "no probe endpoint"}
        print(f"{meta['record'][:34]:<34} {'--':>7}  {'':>6}  no endpoint")
        continue
    p = probe(url)
    probes[meta["key"]] = p
    mark = "OK" if p["ok"] else ("ERR" if p["status"] else "NO NET")
    print(f"{meta['record'][:34]:<34} {str(p['status']):>7}  {p['ms']:>6}  {mark}")

_ok = sum(1 for p in probes.values() if p.get("ok"))
print(f"\n  {_ok} of {len(probes)} records resolved")

---
## 8 · Acquire

Three routes, tried in this order:

1. **Already on disk** — `raw/<key>/` has something in it.
2. **Already mounted** — on Kaggle, the dataset is under `/kaggle/input/`; it is used in place and
   never copied, because a 2 GB copy is a 2 GB waste.
3. **Download** — Kaggle CLI for Kaggle records, plain HTTPS for Zenodo.

Mendeley is the exception: the API is open, the bytes are not. The notebook prints the page, the
file names and the sizes, and waits.

In [ ]:
def find_kaggle_input(slug):
    root = Path("/kaggle/input")
    if not root.is_dir():
        return None
    tail = slug.split("/")[-1].lower()
    for p in root.iterdir():
        if p.is_dir() and (p.name.lower() == tail or tail in p.name.lower()
                           or p.name.lower() in tail):
            return p
    return None

def kaggle_credentials():
    cfg = Path.home() / ".kaggle" / "kaggle.json"
    if cfg.exists():
        return True
    user = os.environ.get("KAGGLE_USERNAME")
    key  = os.environ.get("KAGGLE_KEY")
    if not (user and key) and IS_COLAB:
        try:
            from google.colab import userdata
            user = userdata.get("KAGGLE_USERNAME")
            key  = userdata.get("KAGGLE_KEY")
        except Exception:
            pass
    if not (user and key) and SETTINGS.get("kaggle_json"):
        try:
            j = json.loads(SETTINGS["kaggle_json"])
            user, key = j["username"], j["key"]
        except Exception:
            pass
    if user and key:
        cfg.parent.mkdir(parents=True, exist_ok=True)
        cfg.write_text(json.dumps({"username": user, "key": key}))
        try:
            os.chmod(cfg, 0o600)
        except OSError:
            pass
        return True
    return False

def download_kaggle(slug, dest):
    if importlib.util.find_spec("kaggle") is None:
        _pip("kaggle")
    if not kaggle_credentials():
        return None, "no Kaggle credentials (Account -> Create New API Token)"
    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    cmd = [sys.executable, "-m", "kaggle", "datasets", "download", "-d", slug, "-p", str(dest)]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        tail = (r.stderr or r.stdout).strip().splitlines()
        return None, (tail[-1][:200] if tail else "kaggle CLI failed")
    return dest, "downloaded"

def download_http(url, dest, name=None):
    import requests
    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=180) as r:
        r.raise_for_status()
        fn = name
        if not fn:
            cd = r.headers.get("content-disposition", "")
            m = re.search(r'filename="?([^";]+)"?', cd)
            fn = m.group(1) if m else url.split("/")[-1].split("?")[0]
        target = dest / fn
        total = int(r.headers.get("content-length") or 0)
        done, last, t0 = 0, 0, time.time()
        with open(target, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                if not chunk:
                    continue
                f.write(chunk)
                done += len(chunk)
                if total and done - last > (64 << 20):
                    last = done
                    print(f"      {100*done/total:5.1f}%  {done/1e6:7.1f}/{total/1e6:.1f} MB"
                          f"  {time.time()-t0:4.0f}s", end="\r")
    print(f"      saved {target.name}  {target.stat().st_size/1e6:.1f} MB" + " " * 20)
    return target

def resolve_source(meta):
    """Return (path, kind). path is None when nothing is available."""
    key = meta["key"]
    local = RAW / key

    if local.exists() and any(local.rglob("*")):
        return local, "local"

    if meta.get("kaggle"):
        found = find_kaggle_input(meta["kaggle"])
        if found:
            return found, "kaggle-input"

    if SETTINGS["download"]:
        if meta.get("kaggle"):
            print(f"      kaggle: {meta['kaggle']}")
            path, msg = download_kaggle(meta["kaggle"], local)
            if path:
                return path, "kaggle-download"
            print(f"      ! {msg}")
        for url in (meta.get("http") or []):
            try:
                print(f"      https: {url[:88]}")
                download_http(url, local)
                return local, "http-download"
            except Exception as e:
                print(f"      ! download failed: {str(e)[:140]}")

    return None, "missing"

print("  acquisition helpers ready")

---
## 9 · Run the acquisition

This is the only cell that can take a long time. With `download = False` it takes milliseconds and
reports what is already on disk.

In [ ]:
sources = {}
for meta in ACTIVE:
    key = meta["key"]
    print(f"\n[{key}] {meta['label']}")
    path, kind = resolve_source(meta)
    sources[key] = {"path": str(path) if path else None, "kind": kind}
    if path:
        print(f"      -> {kind}: {path}")
    else:
        print("      -> NOT PRESENT")
        if meta.get("mendeley_files"):
            print(f"      open : {meta['page']}")
            print("      files: " + "; ".join(meta["mendeley_files"]))
            print(f"      drop : {RAW / key}/")
        elif meta.get("kaggle"):
            print(f"      add as Kaggle input: {meta['kaggle']}")
        elif meta.get("http"):
            print(f"      direct: {meta['http'][0]}")

---
## 10 · Extract

Archives are unpacked **once** into `extracted/<key>/<archive-name>/`. Keeping the archive name in
the path matters: Silva's labels are only in the archive names (`miner_img_xml.zip`,
`rust_xml_image.zip`), and the folder fallback finds them there.

If the source is already a directory of images (a Kaggle input), nothing is copied.

In [ ]:
ARCHIVE_SUFFIXES = (".zip", ".tar", ".tar.gz", ".tgz", ".rar", ".7z")

def is_archive(p):
    n = p.name.lower()
    return any(n.endswith(s) for s in ARCHIVE_SUFFIXES)

def extract_one(arc, dst):
    name = arc.name.lower()
    dst = Path(dst)
    try:
        if name.endswith(".zip"):
            with zipfile.ZipFile(arc) as z:
                z.extractall(dst)
        elif name.endswith((".tar.gz", ".tgz", ".tar")):
            with tarfile.open(arc) as t:
                t.extractall(dst)
        elif name.endswith((".rar", ".7z")):
            tool = next((t for t in ("unrar", "7z", "bsdtar") if shutil.which(t)), None)
            if tool is None:
                print(f"      ! no unrar/7z available for {arc.name}")
                return False
            if tool == "unrar":
                subprocess.run(["unrar", "x", "-o+", "-idq", str(arc), str(dst) + "/"], check=False)
            elif tool == "7z":
                subprocess.run(["7z", "x", "-y", f"-o{dst}", str(arc)], check=False)
            else:
                subprocess.run(["bsdtar", "xf", str(arc), "-C", str(dst)], check=False)
        else:
            return False
    except Exception as e:
        print(f"      ! extract failed {arc.name}: {str(e)[:140]}")
        return False
    return True

def extract_tree(src, key):
    src = Path(src)
    archives = [p for p in src.rglob("*") if p.is_file() and is_archive(p)]
    if not archives:
        return src, 0
    dst_root = EXTRACTED / key
    dst_root.mkdir(parents=True, exist_ok=True)
    done = 0
    for arc in archives:
        stem = re.sub(r"[^A-Za-z0-9._-]+", "_", arc.name)
        target = dst_root / stem
        if target.exists() and any(target.rglob("*")):
            done += 1
            continue
        target.mkdir(parents=True, exist_ok=True)
        print(f"      unpack {arc.name} -> {target.name}/")
        if extract_one(arc, target):
            done += 1
    return dst_root, done

roots = {}
for meta in ACTIVE:
    key = meta["key"]
    info = sources.get(key, {})
    if not info.get("path"):
        roots[key] = None
        continue
    root, n = extract_tree(info["path"], key)
    roots[key] = root
    print(f"  {key:<16} {n:>2} archive(s)  ->  {root}")

---
## 11 · Count — the number the report quotes

No filtering beyond the file extension. A `.txt`, a `.csv`, an `.xml` and a `.json` are **not**
images and are not counted; a `.jpg` is counted even if it is corrupt, and the corrupt ones are
reported separately as `unreadable`.

In [ ]:
results = {}
print(f"{'dataset':<26} {'images':>9}  {'unlabelled':>10}  {'classes':>7}  status")
print("-" * 78)

for meta in ACTIVE:
    key = meta["key"]
    root = roots.get(key)
    base = {k: meta.get(k) for k in ("label", "record", "tier", "role", "claim_total")}
    if root is None:
        results[key] = {**base, "status": "NOT PRESENT", "images": 0,
                        "by_class": {}, "by_split": {}, "by_split_class": {},
                        "by_ext": {}, "dimensions": [], "groups": None,
                        "source_kind": sources.get(key, {}).get("kind", "missing")}
        print(f"{meta['label'][:26]:<26} {'-':>9}  {'-':>10}  {'-':>7}  NOT PRESENT")
        continue

    stats = count_tree(root, meta, SETTINGS["dimension_sample"])
    results[key] = {**base, **stats,
                    "status": "counted" if stats["images"] else "EMPTY",
                    "source_kind": sources.get(key, {}).get("kind", "local"),
                    "root": str(root)}
    unl = stats["by_class"].get("unlabelled", 0)
    print(f"{meta['label'][:26]:<26} {stats['images']:>9,}  {unl:>10,}  "
          f"{len(stats['by_class']):>7}  {results[key]['status']}")

_counted = sum(1 for r in results.values() if r["status"] == "counted")
_total = sum(r["images"] for r in results.values() if r["status"] == "counted")
print(f"\n  {_counted} dataset(s) counted, {_total:,} image files in total")
print("  (mirrors are included here; the verdict cell removes them from the totals)")

---
## 12 · Verify — observed against claimed

The tolerance is `max(5 images, 1 %)`. A six-image difference on 58,555 is a **MATCH with a note**,
not a failure: it is the paper being six images wrong, and the notebook says so. A 400-image
difference is a **MISMATCH** and is treated as a finding.

In [ ]:
def verify(meta, res):
    if res["status"] != "counted":
        return {"verdict": res["status"], "delta": None, "tolerance": None,
                "per_class": {}}
    observed = res["images"]
    claimed = meta.get("claim_total")
    if claimed is None:
        return {"verdict": "NO CLAIM", "delta": None, "tolerance": None, "per_class": {}}
    delta = observed - claimed
    tol = max(5, int(round(0.01 * claimed)))
    if delta == 0:
        verdict = "MATCH"
    elif abs(delta) <= tol:
        verdict = "CLOSE"
    else:
        verdict = "MISMATCH"
    per_class = {}
    for cls, exp in (meta.get("expected") or {}).items():
        if exp is None:
            continue
        obs = res["by_class"].get(cls, 0)
        per_class[cls] = {"expected": exp, "observed": obs, "delta": obs - exp}
    return {"verdict": verdict, "delta": delta, "tolerance": tol, "per_class": per_class}

verification = {}
print(f"{'dataset':<26} {'claimed':>8} {'observed':>9} {'delta':>7}  verdict")
print("-" * 70)
for meta in ACTIVE:
    key = meta["key"]
    v = verify(meta, results[key])
    verification[key] = v
    claimed = meta.get("claim_total")
    observed = results[key]["images"] if results[key]["status"] == "counted" else None
    d = v["delta"]
    print(f"{meta['label'][:26]:<26} "
          f"{(f'{claimed:,}' if claimed else '-'):>8} "
          f"{(f'{observed:,}' if observed is not None else '-'):>9} "
          f"{(f'{d:+,}' if d is not None else '-'):>7}  {v['verdict']}")

print("\nPer-class detail (only where the publisher publishes a per-class figure):")
for key, v in verification.items():
    if not v["per_class"]:
        continue
    label = results[key]["label"]
    print(f"  {label}")
    for cls, row in v["per_class"].items():
        flag = "ok" if row["delta"] == 0 else f"{row['delta']:+d}"
        print(f"    {cls:<12} expected {row['expected']:>7,}  observed {row['observed']:>7,}  {flag}")

n_counted  = sum(1 for r in results.values() if r["status"] == "counted")
n_images   = sum(r["images"] for r in results.values() if r["status"] == "counted")
n_mismatch = sum(1 for v in verification.values() if v["verdict"] == "MISMATCH")
n_missing  = len(results) - n_counted

---
## 13 · Hashes and dimensions

The SHA-256 of each archive is the reproducibility artefact: if two people download the same record
and get different bytes, one of them is not looking at the same dataset. Dimensions are sampled, not
exhaustive, and the sample size is stated.

In [ ]:
hashes = {}
print(f"{'archive':<52} {'MB':>8}  sha256")
print("-" * 92)
for meta in ACTIVE:
    key = meta["key"]
    local = RAW / key
    if not local.exists():
        continue
    for arc in sorted(local.rglob("*")):
        if not (arc.is_file() and is_archive(arc)):
            continue
        h = sha256(arc)
        hashes[str(arc.relative_to(BASE))] = {"sha256": h,
                                              "bytes": arc.stat().st_size}
        print(f"{arc.name[:52]:<52} {arc.stat().st_size/1e6:>8.1f}  {h[:32]}...")

if not hashes:
    print("  no local archives to hash (downloads are mounted, or nothing was fetched)")

print("\nDimension sample:")
for key, res in results.items():
    if res["status"] != "counted" or not res.get("dimensions"):
        continue
    dims = res["dimensions"]
    widths  = sorted(w for w, _ in dims)
    heights = sorted(h for _, h in dims)
    med = (widths[len(widths)//2], heights[len(heights)//2])
    mn  = (widths[0], heights[0])
    mx  = (widths[-1], heights[-1])
    print(f"  {res['label'][:24]:<24} n={len(dims):<4} median={med[0]}x{med[1]}  "
          f"min={mn[0]}x{mn[1]}  max={mx[0]}x{mx[1]}  unreadable={res.get('unreadable', 0)}")

---
## 14 · Optional — perceptual-hash duplicates

This is the input to the leakage-safe split, and it is the question the papers do not answer:
**how many of these images are near-copies?**

It is off by default because it opens every image. Turn it on in SETTINGS for the sets that matter
(Uganda, RoCoLe, Ethiopian). A high duplicate rate is not a bug in the dataset — it is the reason a
random split is invalid.

In [ ]:
phash_results = {}

if SETTINGS["run_phash"]:
    if importlib.util.find_spec("imagehash") is None:
        _pip("imagehash")
    import imagehash

    def phash_duplicates(root, threshold=6, limit=None):
        seen = []
        for p in Path(root).rglob("*"):
            if not p.is_file() or p.suffix.lower() not in IMAGE_EXT:
                continue
            try:
                with Image.open(p) as im:
                    h = imagehash.phash(im.convert("RGB"))
            except Exception:
                continue
            seen.append((h, p))
            if limit and len(seen) >= limit:
                break
        clusters = []
        for h, p in seen:
            for c in clusters:
                if h - c["hash"] <= threshold:
                    c["n"] += 1
                    break
            else:
                clusters.append({"hash": h, "n": 1, "example": p.name})
        return {"images": len(seen), "clusters": len(clusters),
                "duplicates": len(seen) - len(clusters),
                "rate": round(1 - len(clusters) / max(1, len(seen)), 4),
                "largest": sorted((c["n"] for c in clusters), reverse=True)[:5]}

    for key in SETTINGS["phash_datasets"]:
        if results.get(key, {}).get("status") != "counted":
            continue
        print(f"  hashing {key} ...")
        phash_results[key] = phash_duplicates(results[key]["root"])
        r = phash_results[key]
        print(f"    {r['images']:,} images -> {r['clusters']:,} clusters, "
              f"{r['duplicates']:,} near-duplicates ({r['rate']*100:.1f} %)")
else:
    print("  skipped — set SETTINGS['run_phash'] = True to run it")

---
## 15 · Dashboard

A single screen that answers the instructor's question. Green means the count agrees with the
publisher, amber means it agrees within tolerance and the difference is explained, red means it does
not agree and that is a finding, grey means the archive has not been opened.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

COLORS = {"healthy": "#2e7d32", "rust": "#c62828", "cercospora": "#6a1b9a",
          "miner": "#ef6c00", "phoma": "#1565c0", "unlabelled": "#9e9e9e",
          "red_spider_mite": "#8d6e63", "mycena": "#00838f", "weevil": "#5d4037"}
BADGE = {"MATCH":      ("#1b5e20", "#e8f5e9"),
         "CLOSE":      ("#e65100", "#fff3e0"),
         "MISMATCH":   ("#b71c1c", "#ffebee"),
         "NOT PRESENT":("#455a64", "#eceff1"),
         "EMPTY":      ("#455a64", "#eceff1"),
         "NO CLAIM":   ("#37474f", "#eceff1")}

counted = [r for r in results.values() if r["status"] == "counted"]
n_counted = len(counted)
# mirrors are counted once: prefer the original records, fall back to the mirror
_non_mirror = [r for r in counted if r["role"] != "mirror"]
_mirror_only = [r for r in counted if r["role"] == "mirror"]
n_images  = sum(r["images"] for r in (_non_mirror or _mirror_only))
n_mismatch = sum(1 for v in verification.values() if v["verdict"] == "MISMATCH")
n_missing  = sum(1 for r in results.values() if r["status"] != "counted")

# ── chart 1: stacked counts ───────────────────────────────────────────────
chart1 = None
if counted:
    labels = [r["label"] for r in counted]
    fig, ax = plt.subplots(figsize=(10.5, max(3.6, 0.62 * len(counted) + 1.6)))
    left = [0] * len(counted)
    for cls in CLASS_ORDER + ["unlabelled"]:
        vals = [r["by_class"].get(cls, 0) for r in counted]
        if not any(vals):
            continue
        ax.barh(labels, vals, left=left, color=COLORS.get(cls, "#607d8b"),
                label=cls, edgecolor="white", linewidth=0.6)
        left = [a + b for a, b in zip(left, vals)]
    for i, total in enumerate(left):
        ax.text(total + max(left) * 0.01, i, f"{int(total):,}",
                va="center", fontsize=9, color="#263238")
    ax.set_xlabel("images (raw, before any preprocessing)")
    ax.set_title("Images per dataset and class", loc="left", fontsize=13, weight="bold")
    ax.invert_yaxis()
    ax.grid(axis="x", alpha=0.25)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.legend(ncol=5, fontsize=8, frameon=False, loc="upper center",
              bbox_to_anchor=(0.5, -0.12))
    fig.tight_layout()
    chart1 = OUT / "dashboard.png"
    fig.savefig(chart1, dpi=150, bbox_inches="tight")
    plt.close(fig)

# ── chart 2: claimed vs observed ─────────────────────────────────────────
chart2 = None
rows2 = [r for r in counted if r.get("claim_total")]
if rows2:
    labels = [r["label"] for r in rows2]
    obs = [r["images"] for r in rows2]
    clm = [r["claim_total"] for r in rows2]
    y = range(len(rows2))
    fig, ax = plt.subplots(figsize=(10.5, max(3.2, 0.62 * len(rows2) + 1.4)))
    ax.barh([i + 0.20 for i in y], clm, height=0.38, color="#cfd8dc",
            label="claimed", edgecolor="white")
    ax.barh([i - 0.20 for i in y], obs, height=0.38, color="#1565c0",
            label="observed", edgecolor="white")
    ax.set_yticks(list(y))
    ax.set_yticklabels(labels)
    ax.invert_yaxis()
    ax.set_xlabel("images")
    ax.set_title("Claimed vs observed", loc="left", fontsize=13, weight="bold")
    ax.grid(axis="x", alpha=0.25)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.legend(frameon=False)
    fig.tight_layout()
    chart2 = OUT / "claimed_vs_observed.png"
    fig.savefig(chart2, dpi=150, bbox_inches="tight")
    plt.close(fig)

# ── HTML table ────────────────────────────────────────────────────────────
def badge(text):
    fg, bg = BADGE.get(text, ("#37474f", "#eceff1"))
    return (f'<span style="background:{bg};color:{fg};padding:2px 8px;'
            f'border-radius:10px;font-size:11px;font-weight:600">{text}</span>')

rows_html = []
for meta in ACTIVE:
    key = meta["key"]
    res = results[key]
    v = verification[key]
    observed = f"{res['images']:,}" if res["status"] == "counted" else "&mdash;"
    claimed = f"{meta['claim_total']:,}" if meta.get("claim_total") else "&mdash;"
    delta = v["delta"]
    if delta is None:
        delta_s = "&mdash;"
    elif delta == 0:
        delta_s = "0"
    else:
        color = "#b71c1c" if abs(delta) > (v["tolerance"] or 0) else "#e65100"
        delta_s = f'<span style="color:{color};font-weight:600">{delta:+,}</span>'
    classes = ", ".join(f"{c} {n:,}" for c, n in
                        sorted(res.get("by_class", {}).items(), key=lambda kv: -kv[1])
                        if c != "unlabelled") or "&mdash;"
    rows_html.append(f"""
      <tr>
        <td><b>{meta['label']}</b><br><span class="sub">{meta['record']}</span></td>
        <td class="c">{meta['tier']}</td>
        <td class="c lic">{meta['licence']}</td>
        <td class="r">{claimed}</td>
        <td class="r"><b>{observed}</b></td>
        <td class="r">{delta_s}</td>
        <td class="c">{badge(v['verdict'])}</td>
        <td class="classes">{classes}</td>
      </tr>""")

html = f"""
<div class="wrap">
  <div class="hdr">
    <div>
      <div class="eyebrow">BunaDoctor &middot; dataset verification</div>
      <div class="title">The count before any preprocessing</div>
      <div class="sub">generated {time.strftime('%d %b %Y, %H:%M UTC', time.gmtime())} &middot; {ENV}</div>
    </div>
  </div>

  <div class="cards">
    <div class="card"><div class="k">datasets counted</div><div class="v">{n_counted}<span>/{len(results)}</span></div></div>
    <div class="card"><div class="k">images counted</div><div class="v">{n_images:,}</div></div>
    <div class="card"><div class="k">mismatches</div><div class="v" style="color:{'#b71c1c' if n_mismatch else '#1b5e20'}">{n_mismatch}</div></div>
    <div class="card"><div class="k">not opened</div><div class="v" style="color:#455a64">{n_missing}</div></div>
  </div>

  <table>
    <thead><tr>
      <th>dataset</th><th class="c">tier</th><th class="c">licence</th><th class="r">claimed</th>
      <th class="r">observed</th><th class="r">&Delta;</th>
      <th class="c">verdict</th><th>classes observed</th>
    </tr></thead>
    <tbody>{''.join(rows_html)}
    </tbody>
  </table>

  <div class="foot">
    A count is a <code>len()</code> over a real directory. <b>NOT PRESENT</b> means the archive was
    not on disk and was not downloaded &mdash; it is never replaced by a published figure.
  </div>
</div>

<style>
  .wrap {{ font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
           color:#263238; max-width:1120px; }}
  .hdr {{ display:flex; justify-content:space-between; align-items:flex-end;
          border-bottom:3px solid #263238; padding-bottom:10px; margin-bottom:16px; }}
  .eyebrow {{ font-size:11px; letter-spacing:.12em; text-transform:uppercase; color:#6d4c41;
              font-weight:700; }}
  .title {{ font-size:24px; font-weight:800; margin-top:2px; }}
  .sub {{ font-size:12px; color:#607d8b; }}
  .cards {{ display:grid; grid-template-columns:repeat(4,1fr); gap:12px; margin-bottom:18px; }}
  .card {{ background:#fafafa; border:1px solid #e0e0e0; border-radius:10px; padding:12px 14px; }}
  .card .k {{ font-size:11px; text-transform:uppercase; letter-spacing:.08em; color:#78909c; }}
  .card .v {{ font-size:26px; font-weight:800; margin-top:2px; }}
  .card .v span {{ font-size:14px; color:#90a4ae; font-weight:500; }}
  table {{ width:100%; border-collapse:collapse; font-size:13px; }}
  th {{ text-align:left; font-size:11px; text-transform:uppercase; letter-spacing:.06em;
        color:#546e7a; border-bottom:2px solid #cfd8dc; padding:8px 6px; }}
  td {{ padding:9px 6px; border-bottom:1px solid #eceff1; vertical-align:top; }}
  tr:hover td {{ background:#fafafa; }}
  .c {{ text-align:center; }} .r {{ text-align:right; font-variant-numeric:tabular-nums; }}
  .lic {{ font-size:11px; color:#546e7a; white-space:nowrap; }}
  .sub {{ font-size:11px; color:#90a4ae; }}
  .classes {{ font-size:12px; color:#455a64; }}
  .foot {{ margin-top:14px; font-size:12px; color:#607d8b; }}
</style>
"""

try:
    from IPython.display import HTML, Image as IPyImage, display
    display(HTML(html))
    if chart1:
        display(IPyImage(filename=str(chart1)))
    if chart2:
        display(IPyImage(filename=str(chart2)))
except Exception:
    print(html)
    print(f"charts: {chart1} {chart2}")

---
## 16 · Export

Nothing in these files is hand-typed. `dataset_counts.json` is what the report's SO1 points at;
`dataset_counts.md` is pasted into the datasheet; the CSV is what a reviewer diffs.

In [ ]:
payload = {
    "generated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "environment": ENV,
    "python": platform.python_version(),
    "note": ("Raw file counts before any preprocessing: no resize, no dedup, "
             "no augmentation, no split."),
    "settings": {k: v for k, v in SETTINGS.items() if k != "kaggle_json"},
    "probes": probes,
    "sources": sources,
    "results": {k: {kk: vv for kk, vv in v.items() if kk != "dimensions"}
                for k, v in results.items()},
    "verification": verification,
    "archive_sha256": hashes,
    "phash": phash_results,
}
(OUT / "dataset_counts.json").write_text(json.dumps(payload, indent=2, default=str))

with open(OUT / "dataset_counts.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["key", "label", "record", "tier", "role", "licence", "status", "claimed",
                "observed", "delta", "verdict", "source_kind", "classes"])
    for meta in ACTIVE:
        r = results[meta["key"]]
        v = verification[meta["key"]]
        w.writerow([meta["key"], meta["label"], meta["record"], meta["tier"], meta["role"],
                    meta["licence"], r["status"], meta.get("claim_total"), r["images"],
                    v["delta"], v["verdict"], r.get("source_kind"),
                    ";".join(f"{c}={n}" for c, n in r.get("by_class", {}).items())])

with open(OUT / "dataset_class_counts.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["key", "label", "split", "class", "images"])
    for meta in ACTIVE:
        r = results[meta["key"]]
        if r["status"] != "counted":
            continue
        if r.get("by_split_class"):
            for split, classes in r["by_split_class"].items():
                for cls, n in classes.items():
                    w.writerow([meta["key"], meta["label"], split, cls, n])
        else:
            for cls, n in r["by_class"].items():
                w.writerow([meta["key"], meta["label"], "unspecified", cls, n])

lines = ["# Dataset counts — raw, before any preprocessing", "",
         f"Generated {payload['generated_utc']} on {ENV}.", "",
         "| Dataset | Tier | Licence | Claimed | Observed | Δ | Verdict | Classes |",
         "|---|---|---|---:|---:|---:|---|---|"]
for meta in ACTIVE:
    r = results[meta["key"]]
    v = verification[meta["key"]]
    obs = f"{r['images']:,}" if r["status"] == "counted" else "—"
    clm = f"{meta['claim_total']:,}" if meta.get("claim_total") else "—"
    d = "—" if v["delta"] is None else f"{v['delta']:+,}"
    classes = ", ".join(f"{c} {n:,}" for c, n in r.get("by_class", {}).items()) or "—"
    lines.append(f"| {meta['label']} | {meta['tier']} | {meta['licence']} | {clm} | **{obs}** | {d} | "
                 f"{v['verdict']} | {classes} |")
lines += ["", "Counts are file counts. No resize, no dedup, no augmentation, no split."]
(OUT / "dataset_counts.md").write_text("\n".join(lines) + "\n")

manifest = OUT / "RUN_MANIFEST.txt"
manifest.write_text("\n".join([
    f"generated_utc : {payload['generated_utc']}",
    f"environment   : {ENV}",
    f"python        : {platform.python_version()}",
    f"datasets      : {len(ACTIVE)} active of {len(DATASETS)}",
    f"counted       : {n_counted}",
    f"images        : {n_images:,}",
    "",
    "packages:",
    *[f"  {m:<12} {importlib.metadata.version(m) if importlib.util.find_spec(m) else 'missing'}"
      for m in ("pillow", "pandas", "requests", "matplotlib", "numpy")],
]))

import zipfile as _zip
bundle = OUT / "buna-verification.zip"
with _zip.ZipFile(bundle, "w", _zip.ZIP_DEFLATED) as z:
    for name in ("dataset_counts.json", "dataset_counts.csv", "dataset_class_counts.csv",
                 "dataset_counts.md", "RUN_MANIFEST.txt",
                 "dashboard.png", "claimed_vs_observed.png"):
        p = OUT / name
        if p.exists():
            z.write(p, name)

for p in sorted(OUT.iterdir()):
    print(f"  {p.name:<28} {p.stat().st_size/1024:>8.1f} KB")

if IS_COLAB:
    print("\n  Colab: use the Files pane (left) -> buna-verify/out -> download")
elif IS_KAGGLE:
    print("\n  Kaggle: the files are under /kaggle/working/buna-verify/out and are saved with the notebook")

---
## 17 · Verdict

The sentences the group is allowed to say out loud, with the numbers filled in from disk.

In [ ]:
training = [r for r in results.values()
            if r["role"] == "training" and r["status"] == "counted"]
mirror   = [r for r in results.values()
            if r["role"] == "mirror" and r["status"] == "counted"]
external = [r for r in results.values()
            if r["role"] == "external" and r["status"] == "counted"]

train_total = sum(r["images"] for r in training)
mirror_total = sum(r["images"] for r in mirror)

print("=" * 78)
print("  THE COUNT, BEFORE ANY PREPROCESSING")
print("=" * 78)
print(f"  datasets counted   : {n_counted} of {len(results)}")
print(f"  images counted     : {n_images:,}")
if training:
    print(f"  training corpus    : {train_total:,}  "
          f"({', '.join(r['label'] for r in training)})")
elif mirror:
    print(f"  training corpus    : {mirror_total:,}  "
          f"(counted from the Kaggle mirror of JMuBEN + JMuBEN2)")
if mirror and training:
    print(f"  mirror (not added) : {mirror_total:,}  "
          f"({', '.join(r['label'] for r in mirror)})")
print(f"  external sets      : {len(external)}  "
      f"({', '.join(r['label'] for r in external) if external else 'none counted'})")
print(f"  mismatches         : {n_mismatch}")

print("\n  What we can say:")
_said = 0
if training or mirror:
    _said += 1
    base = train_total or mirror_total
    print(f"  {_said}. The five-class training corpus is {base:,} images, counted from files.")
for key, v in verification.items():
    if v["verdict"] in ("CLOSE", "MISMATCH") and results[key].get("claim_total"):
        _said += 1
        print(f"  {_said}. {results[key]['label']}: the publisher says "
              f"{results[key]['claim_total']:,}; the files say "
              f"{results[key]['images']:,} ({v['delta']:+,}).")
if external:
    _said += 1
    print(f"  {_said}. {len(external)} external set(s) counted, "
          f"{sum(r['images'] for r in external):,} images, none of them used for training.")
if results.get("ethiopian", {}).get("status") == "counted":
    _said += 1
    tc = results["ethiopian"]["by_split"].get("test", 0)
    print(f"  {_said}. The Ethiopian test split is {tc:,} images — "
          f"the only Cercospora set we can verify and hold out.")
if _said == 0:
    print("  (nothing counted yet) — set SETTINGS['download'] = True, or add a")
    print("  Kaggle input, then re-run from cell 8. The 11 probes above still")
    print("  prove that every record exists on the day this was run.")

missing = [m for m in ACTIVE if results[m["key"]]["status"] != "counted"]
if missing:
    print(f"\n  Still to open ({len(missing)}): "
          f"{', '.join(m['label'] for m in missing)}")
    print("  These are T3 until this notebook counts them. Do not quote them as observed.")
else:
    print("\n  Every record in the registry was opened and counted.")

print("\n  Reproduce with: notebooks/00-dataset-verification.ipynb")
print("  Evidence table : docs/12-DATASET-VERIFICATION.md")
print("=" * 78)

---
## What the numbers mean — and what they do not

**A count is not a quality claim.** 58,549 images is a count. It does not say that 58,549 *distinct
plants* were photographed, that the labels are correct, or that the images are not augmented copies
of one another. JMuBEN's own publishers state that the release is cropped, resized and
rotation/flipped, and that the originals were never published — which is why the leakage-safe split
exists and why §14 of this notebook is not optional for the headline result.

**The Ethiopian set is the only Cercospora set we can verify and hold out.** It is a self-uploaded
CC0 Kaggle record with no paper and no named annotator. Its `test/` split is balanced at 300 per
class, which is evidence of **curation**, not of a random field sample — so its accuracy is an upper
bound and the caveat travels beside the number.

**A Kaggle mirror is not a source.** Mirrors relabel the licence and most get it wrong. The DOI in
the registry is the citation; the mirror is only a faster download. One mirror is worse than wrong:
the augmented Uganda release `chelangatspecioza/coffeedataset34` is **CC BY-NC-SA 4.0**, so it
cannot be used to train a product that is meant to become a company. The Mendeley original is
CC BY 4.0. If BunaDoctor is to be a startup, **train on the original, not the augmented mirror** —
and say which one you used.

**`unlabelled` is a finding.** If a dataset lands with 3,894 images and 0 class labels, the total is
verified and the per-class split is not. Say so; do not invent the split from the paper.

### The four-tier evidence scale

| Tier | What it means | How to raise it |
|---|---|---|
| **T1** | two **independent** releases agree | count a second publisher |
| **T2** | one publisher file manifest, every file enumerated | count a second publisher |
| **T3** | published figure, archive not opened | run this notebook on the archive |
| **T4** | claimed only, no file manifest | find a manifest |

Silva is **T2**, not T1: its second release is the same author's own Kaggle upload.

---
## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| Every dataset says **NOT PRESENT** | `download` is `False` and nothing is mounted | set `SETTINGS["download"] = True`, or add the dataset as a Kaggle input |
| `no Kaggle credentials` | Colab has no token | kaggle.com → Account → **Create New API Token** → upload `kaggle.json` |
| `403` from Kaggle | you accepted no licence on the dataset page | open the dataset page once while logged in, then retry |
| `no unrar/7z available` | Saposoa is a `.rar` | Colab installs `unrar-free` automatically; on Kaggle use the Kaggle mirror or `pip install patool` |
| A class shows as `unlabelled` | labels live in an annotation file, not in folder names | read the archive's `annotations/` folder; add a `filename_rules` entry to that registry record |
| Count is 3–6 images off | the publisher's own table is off | keep the observed count and print the delta — that is the finding |
| `MISMATCH` | the archive is a partial mirror | check `source_kind`; a mirror is not the record |
| `out of memory` | `run_phash` on 60k images | run it per dataset, or leave it off |

### If you only have five minutes

Run the notebook with `download = False`. You will see the self-test pass, the probes resolve, and
the dashboard explain exactly what is missing. That is a complete, honest result — and it is the
result to show if the network is down.

---
## Next step

The count is the **input** to the split, not the split. Once the counts above are accepted:

```bash
python scripts/prepare_data.py --config data/splits/corpus.yaml --audit
```

`prepare_data.py` performs the leakage-safe grouped split and asserts four invariants (T-03…T-06).
It refuses to write a manifest that fails the audit.

| Document | Why |
|---|---|
| `docs/12-DATASET-VERIFICATION.md` | the full evidence table, the five discrepancies, the licence traps |
| `data/datasheet.md` | the datasheet, §9.2 / §9.6 |
| `data/dataset-sources.md` | the registry with every DOI and every rejected record |
| `notebooks/README.md` | the download table and the step-by-step run guide |

---

*Addis Ababa University · Department of Computer Science · FYP I 2026/27 · Hayredin Mohammed (UGR/2361/16)*